In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.features import carregar_diario, criar_features, dividir_treino_teste
from src.metricas import metricas

diario = carregar_diario("../data/btc_diario.csv")
X, y = criar_features(diario)
X.head()

,close_lag_1,close_lag_7,close_lag_30,volume_lag_1,retorno_1d
data,,,,,
2018-01-31,10419.607123,10074.357813,10035.471708,769.656610,0.021486
2018-02-01,10383.965621,10036.525031,9936.473297,528.576411,-0.003421
2018-02-02,10337.741348,10006.311676,10016.518417,658.393851,-0.004452
2018-02-03,10261.364075,10064.542595,10115.574889,202.328644,-0.007388
2018-02-04,10327.962017,10106.087001,9925.471370,20.922295,0.006490


In [2]:
FIM_TREINO = "2024-12-31"  # a confirmar

X_treino, X_teste = dividir_treino_teste(X, FIM_TREINO)
y_treino, y_teste = dividir_treino_teste(y, FIM_TREINO)

print(len(X_treino), len(X_teste))

2527 680


In [3]:
# Conferência: o naive com as variáveis de Lucas deve repetir o resultado do 03_baselines
metricas(y_teste, X_teste["close_lag_1"])

{'MAE': 80.42923186342836,
 'RMSE': 99.68150424153964,
 'MAPE_%': 0.49818302538571946}

In [4]:
from sklearn.linear_model import LinearRegression

# Cria o modelo 
modelo_linear = LinearRegression()

# Treino: o modelo aprende com as pistas do passado (X_treino) e as respostas certas (y_treino)
modelo_linear.fit(X_treino, y_treino)

# Previsão: o modelo recebe só as pistas do futuro (X_teste) e chuta o preço do dia seguinte
previsoes_linear = modelo_linear.predict(X_teste)

# Avaliação: compara o preço real (y_teste) com o previsto, em MAE, RMSE e MAPE
metricas(y_teste, previsoes_linear)

{'MAE': 82.07643813771887,
 'RMSE': 101.92254128843297,
 'MAPE_%': 0.5074037716888118}

In [5]:
from sklearn.linear_model import Ridge

# Cria o modelo Ridge; alpha controla a força do freio nos pesos (valor inicial 1.0)
modelo_ridge = Ridge(alpha=1.0)

# Treino: aprende com as pistas do passado e as respostas certas
modelo_ridge.fit(X_treino, y_treino)

# Previsão: chuta o preço do dia seguinte só com as pistas do futuro
previsoes_ridge = modelo_ridge.predict(X_teste)

# Avaliação: compara o preço real com o previsto
metricas(y_teste, previsoes_ridge)

{'MAE': 82.07228748617405,
 'RMSE': 101.92458423618248,
 'MAPE_%': 0.5073471011609054}

In [6]:
# Confere se o teste tem preços fora da faixa vista no treino
print(X_treino["close_lag_1"].max(), X_teste["close_lag_1"].max())

15137.667758223422 20098.91023734187


In [7]:
from sklearn.neighbors import KNeighborsRegressor

# Cria o modelo: vai olhar os 5 dias mais parecidos do treino
modelo_knn = KNeighborsRegressor(n_neighbors=5)

# Treino: o KNN guarda os dias do treino com as respostas
modelo_knn.fit(X_treino, y_treino)

# Previsão: para cada dia do teste, faz a média do que veio depois dos 5 dias mais parecidos
previsoes_knn = modelo_knn.predict(X_teste)

# Avaliação
metricas(y_teste, previsoes_knn)

{'MAE': 1789.3636273852826,
 'RMSE': 2497.9958326018304,
 'MAPE_%': 9.871396633940751}

In [8]:
# Novo alvo: variação do preço (amanhã - hoje), que não depende do nível do preço
variacao_treino = y_treino - X_treino["close_lag_1"]

# Treino: o KNN aprende a prever a variação, e não o preço
modelo_knn_var = KNeighborsRegressor(n_neighbors=5)
modelo_knn_var.fit(X_treino, variacao_treino)

# Previsão: o KNN chuta a variação; somamos o preço de hoje para voltar a ter o preço de amanhã
variacao_prevista = modelo_knn_var.predict(X_teste)
previsoes_knn_var = X_teste["close_lag_1"] + variacao_prevista

# Avaliação: continua comparando preços em dólares
metricas(y_teste, previsoes_knn_var)

{'MAE': 92.02535321355712,
 'RMSE': 115.42664873592015,
 'MAPE_%': 0.5719969699470043}

In [9]:
# Tabela de previsões no formato do previsoes_exemplo.csv
previsoes = pd.DataFrame({
    "data": X_teste.index + pd.Timedelta(days=1),  # dia do preço previsto (amanhã)
    "real": y_teste.values,                         # preço real desse dia
    "naive": X_teste["close_lag_1"].values,         # baseline: preço de hoje
    "regressao_linear": previsoes_linear,
    "ridge": previsoes_ridge,
    "knn": previsoes_knn_var.values,                # KNN que prevê a variação
})

# Conferência: o naive da tabela deve dar MAE 80,43
print(metricas(previsoes["real"], previsoes["naive"]))
print(previsoes.shape)

previsoes.to_csv("../data/previsoes_modelos.csv", index=False)
previsoes.head()

{'MAE': 80.42923186342836, 'RMSE': 99.68150424153964, 'MAPE_%': 0.49818302538571946}
(680, 6)


,data,real,naive,regressao_linear,ridge,knn
0,2025-01-02,12710.574270,12562.073652,12564.044756,12559.319562,12555.138601
1,2025-01-03,12864.586704,12710.574270,12714.976730,12709.082757,12752.789967
2,2025-01-04,12905.482348,12864.586704,12867.521515,12862.325949,12859.415070
3,2025-01-05,12915.088162,12905.482348,12903.135781,12902.897622,12988.762672
4,2025-01-06,13052.903824,12915.088162,12912.285567,12912.992797,12904.802321
